# SpecSafe V5 Kaggle Qwen Preflight

## Purpose

This notebook qualifies the approved Qwen model pair for a later trace-collection notebook. It performs **preflight only**:

- resolves immutable Hugging Face revisions;
- records restricted environment metadata;
- rejects a Kaggle GPU architecture unsupported by the installed PyTorch build;
- proves exact tokenizer compatibility;
- proves one finite logits forward pass for each model;
- writes one machine-readable qualification result.

It does **not** collect traces, score policies, run a benchmark, publish data, or make a production claim.

## Before running in Kaggle

1. Create a Kaggle Notebook and upload this `.ipynb` file.
2. In Notebook Settings, enable **Internet** and select **GPU T4 x2**.
3. After this notebook PR is merged, copy the exact `main` commit SHA into the configuration cell below.
4. Run all cells in order.
5. Download `/kaggle/working/specsafe_v5_qwen_preflight_result.json` and retain it for the next SpecSafe slice.

No token, API key, customer prompt, or private data is required for this public-model preflight.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import sys
from dataclasses import dataclass
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

import torch
import transformers
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# Required after this notebook PR merges. Do not run with this value unchanged.
SOURCE_COMMIT_SHA = "RECORD_MAIN_COMMIT_SHA_AFTER_PR_MERGE"

PREFLIGHT_ID = "v5-qwen-same-tokenizer-preflight-v1"
RESULT_SCHEMA_VERSION = "specsafe-kaggle-preflight-result-v1"
DRAFT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
TARGET_MODEL_ID = "Qwen/Qwen2.5-1.5B"
SEED = 20260708
MINIMUM_TRANSFORMERS_VERSION = (4, 37, 0)
OUTPUT_PATH = Path("/kaggle/working/specsafe_v5_qwen_preflight_result.json")

# Self-authored probes only. They test tokenizer identity; they are not trace-collection data.
TOKENIZER_PROBES = (
    "SpecSafe verifies one candidate position at a time.",
    "def add(left, right):\n    return left + right",
    "A small lamp sat beside the book.",
    "Numbers: 1, 2, 3, 5, 8.",
    "Short multilingual probe: café 東京 مرحبا.",
    "Symbols: []{}() <> +-= / * ! ?",
)


class PreflightFailure(RuntimeError):
    """Stops the notebook after retaining a bounded failure result."""

    def __init__(self, code: str, message: str) -> None:
        super().__init__(message)
        self.code = code


@dataclass(frozen=True)
class ModelRevision:
    """Resolved immutable model-hub identity."""

    model_id: str
    revision: str
    license: str | None


def canonical_json(value: dict[str, Any]) -> str:
    """Produce deterministic, downloadable JSON."""

    return json.dumps(value, indent=2, sort_keys=True) + "\n"


def sha256_file(path: Path) -> str:
    """Return the SHA-256 for one retained artifact."""

    return hashlib.sha256(path.read_bytes()).hexdigest()


def safe_error_message(error: BaseException) -> str:
    """Keep retained failures bounded and avoid environment dumps."""

    return f"{type(error).__name__}: {str(error)[:400]}"


def package_version_tuple(raw_version: str) -> tuple[int, int, int]:
    """Compare the numeric prefix of a package version without new dependencies."""

    numeric_parts: list[int] = []
    for part in raw_version.split("."):
        digits = "".join(character for character in part if character.isdigit())
        if not digits:
            break
        numeric_parts.append(int(digits))
        if len(numeric_parts) == 3:
            break
    while len(numeric_parts) < 3:
        numeric_parts.append(0)
    return tuple(numeric_parts)


def require_kaggle_gpu() -> None:
    """Require the declared GPU boundary for model-loading preflight."""

    if not torch.cuda.is_available():
        raise PreflightFailure(
            "gpu_unavailable",
            "Kaggle GPU is unavailable. Enable a GPU accelerator before running preflight.",
        )


def require_supported_gpu_architecture() -> dict[str, Any]:
    """Reject Kaggle GPUs unsupported by the installed PyTorch CUDA build."""

    capability_major, capability_minor = torch.cuda.get_device_capability(0)
    active_architecture = f"sm_{capability_major}{capability_minor}"
    supported_architectures = sorted(torch.cuda.get_arch_list())
    if not supported_architectures:
        raise PreflightFailure(
            "gpu_architecture_unverified",
            "Installed PyTorch did not report supported CUDA architectures.",
        )
    if active_architecture not in supported_architectures:
        raise PreflightFailure(
            "gpu_architecture_unsupported",
            "Active Kaggle GPU architecture "
            f"{active_architecture} is unsupported by the installed PyTorch build. "
            "Select a compatible GPU accelerator and rerun unchanged.",
        )
    return {
        "active_gpu_architecture": active_architecture,
        "torch_supported_gpu_architectures": supported_architectures,
    }


def require_source_commit_sha() -> str:
    """Require an explicit source commit for reproducibility."""

    if SOURCE_COMMIT_SHA == "RECORD_MAIN_COMMIT_SHA_AFTER_PR_MERGE":
        raise PreflightFailure(
            "source_commit_sha_missing",
            "Record the merged main commit SHA in SOURCE_COMMIT_SHA before running preflight.",
        )
    if len(SOURCE_COMMIT_SHA) < 7:
        raise PreflightFailure(
            "source_commit_sha_invalid",
            "SOURCE_COMMIT_SHA must contain at least seven commit characters.",
        )
    return SOURCE_COMMIT_SHA


def require_supported_transformers() -> str:
    """Require a Transformers release that knows the Qwen2 architecture."""

    resolved_version = transformers.__version__
    if package_version_tuple(resolved_version) < MINIMUM_TRANSFORMERS_VERSION:
        raise PreflightFailure(
            "transformers_version_unsupported",
            "Qwen2.5 requires transformers >= 4.37.0 for this preflight.",
        )
    return resolved_version


def resolve_model_revision(api: HfApi, model_id: str) -> ModelRevision:
    """Resolve a model hub repository to one immutable commit SHA."""

    try:
        info = api.model_info(model_id)
    except Exception as error:
        raise PreflightFailure(
            "model_metadata_unavailable",
            f"Could not resolve public model metadata for {model_id}: {safe_error_message(error)}",
        ) from error

    if not info.sha:
        raise PreflightFailure(
            "model_revision_missing",
            f"Model hub did not return an immutable revision for {model_id}.",
        )
    return ModelRevision(model_id=model_id, revision=info.sha, license=info.cardData.get("license"))


def additional_special_token_ids(tokenizer: Any) -> list[int]:
    """Derive additional-special-token IDs without relying on legacy tokenizer APIs."""

    additional_tokens = list(getattr(tokenizer, "additional_special_tokens", []) or [])
    resolved_ids = tokenizer.convert_tokens_to_ids([str(token) for token in additional_tokens])
    if isinstance(resolved_ids, int):
        resolved_ids = [resolved_ids]
    return [int(token_id) for token_id in resolved_ids]


def special_token_ids(tokenizer: Any) -> dict[str, Any]:
    """Capture all standard special-token IDs required by this compatibility gate."""

    return {
        "bos_token_id": tokenizer.bos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "unk_token_id": tokenizer.unk_token_id,
        "pad_token_id": tokenizer.pad_token_id,
        "sep_token_id": tokenizer.sep_token_id,
        "cls_token_id": tokenizer.cls_token_id,
        "mask_token_id": tokenizer.mask_token_id,
        "additional_special_token_ids": additional_special_token_ids(tokenizer),
    }


def compare_tokenizers(draft_tokenizer: Any, target_tokenizer: Any) -> dict[str, Any]:
    """Prove token-space identity before any model trace work is allowed."""

    probe_results = []
    for probe in TOKENIZER_PROBES:
        draft_ids = draft_tokenizer.encode(probe, add_special_tokens=False)
        target_ids = target_tokenizer.encode(probe, add_special_tokens=False)
        probe_results.append(
            {
                "probe_sha256": hashlib.sha256(probe.encode("utf-8")).hexdigest(),
                "token_count": len(draft_ids),
                "token_ids_match": draft_ids == target_ids,
            }
        )

    checks = {
        "tokenizer_class_match": type(draft_tokenizer).__name__ == type(target_tokenizer).__name__,
        "vocabulary_size_match": len(draft_tokenizer) == len(target_tokenizer),
        "token_to_id_mapping_match": draft_tokenizer.get_vocab() == target_tokenizer.get_vocab(),
        "special_token_map_match": draft_tokenizer.special_tokens_map
        == target_tokenizer.special_tokens_map,
        "special_token_id_match": special_token_ids(draft_tokenizer)
        == special_token_ids(target_tokenizer),
        "probe_token_ids_match": all(item["token_ids_match"] for item in probe_results),
    }
    return {
        "passed": all(checks.values()),
        "checks": checks,
        "draft_tokenizer_class": type(draft_tokenizer).__name__,
        "target_tokenizer_class": type(target_tokenizer).__name__,
        "draft_vocabulary_size": len(draft_tokenizer),
        "target_vocabulary_size": len(target_tokenizer),
        "probe_results": probe_results,
    }


def choose_dtype() -> torch.dtype:
    """Select the smallest supported safe dtype for one Kaggle GPU preflight."""

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def test_logits_access(
    *,
    model_id: str,
    revision: str,
    tokenizer: Any,
    dtype: torch.dtype,
) -> dict[str, Any]:
    """Load one model at a time and prove that finite logits are available."""

    device = torch.device("cuda")
    input_ids = tokenizer.encode(TOKENIZER_PROBES[0], add_special_tokens=False)
    if not input_ids:
        raise PreflightFailure(
            "empty_probe_encoding", "Tokenizer produced an empty probe encoding."
        )

    model = None
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            revision=revision,
            torch_dtype=dtype,
            trust_remote_code=False,
        )
        model.to(device)
        model.eval()
        tensor = torch.tensor([input_ids], dtype=torch.long, device=device)
        with torch.inference_mode():
            logits = model(input_ids=tensor).logits
        finite = bool(torch.isfinite(logits).all().item())
        expected_vocab_size = len(tokenizer)
        observed_vocab_size = int(logits.shape[-1])
        if not finite or observed_vocab_size != expected_vocab_size:
            raise PreflightFailure(
                "logits_access_failed",
                "Model logits are non-finite or incompatible with the tokenizer vocabulary.",
            )
        return {
            "passed": True,
            "model_id": model_id,
            "revision": revision,
            "dtype": str(dtype),
            "logits_shape": list(logits.shape),
            "observed_vocab_size": observed_vocab_size,
        }
    except PreflightFailure:
        raise
    except Exception as error:
        raise PreflightFailure(
            "model_load_or_logits_failure",
            f"Could not load {model_id} or access finite logits: {safe_error_message(error)}",
        ) from error
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


def environment_metadata() -> dict[str, Any]:
    """Record only bounded reproducibility metadata; never dump the full environment."""

    cuda_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
    gpu_architecture = None
    supported_architectures: list[str] = []
    if cuda_available:
        capability_major, capability_minor = torch.cuda.get_device_capability(0)
        gpu_architecture = f"sm_{capability_major}{capability_minor}"
        supported_architectures = sorted(torch.cuda.get_arch_list())
    return {
        "python_version": sys.version.split()[0],
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "huggingface_hub_version": version("huggingface_hub"),
        "cuda_available": cuda_available,
        "cuda_version": torch.version.cuda,
        "gpu_name": gpu_name,
        "gpu_architecture": gpu_architecture,
        "torch_supported_gpu_architectures": supported_architectures,
        "kaggle_kernel_run_type": os.getenv("KAGGLE_KERNEL_RUN_TYPE"),
        "kaggle_url_base": os.getenv("KAGGLE_URL_BASE"),
    }


def write_result(payload: dict[str, Any]) -> Path:
    """Write one canonical retained preflight result in Kaggle working storage."""

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(canonical_json(payload), encoding="utf-8")
    return OUTPUT_PATH

In [ ]:
set_seed(SEED)

result: dict[str, Any] = {
    "schema_version": RESULT_SCHEMA_VERSION,
    "preflight_id": PREFLIGHT_ID,
    "preflight_status": "fails_kaggle_preflight",
    "trace_collection_allowed": False,
    "trace_collection_performed": False,
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "source_commit_sha": None,
    "draft_model": {"model_id": DRAFT_MODEL_ID, "revision": None, "license": None},
    "target_model": {"model_id": TARGET_MODEL_ID, "revision": None, "license": None},
    "tokenizer_compatibility": None,
    "draft_logits_access": None,
    "target_logits_access": None,
    "environment": environment_metadata(),
    "failure": None,
}

try:
    source_commit_sha = require_source_commit_sha()
    require_kaggle_gpu()
    require_supported_transformers()

    api = HfApi()
    draft_revision = resolve_model_revision(api, DRAFT_MODEL_ID)
    target_revision = resolve_model_revision(api, TARGET_MODEL_ID)
    result["source_commit_sha"] = source_commit_sha
    result["draft_model"] = {
        "model_id": draft_revision.model_id,
        "revision": draft_revision.revision,
        "license": draft_revision.license,
    }
    result["target_model"] = {
        "model_id": target_revision.model_id,
        "revision": target_revision.revision,
        "license": target_revision.license,
    }

    draft_tokenizer = AutoTokenizer.from_pretrained(
        draft_revision.model_id,
        revision=draft_revision.revision,
        trust_remote_code=False,
        use_fast=True,
    )
    target_tokenizer = AutoTokenizer.from_pretrained(
        target_revision.model_id,
        revision=target_revision.revision,
        trust_remote_code=False,
        use_fast=True,
    )
    tokenizer_result = compare_tokenizers(draft_tokenizer, target_tokenizer)
    result["tokenizer_compatibility"] = tokenizer_result
    if not tokenizer_result["passed"]:
        raise PreflightFailure(
            "tokenizer_incompatible",
            "Draft and target tokenizers are not exactly compatible. Trace collection is blocked.",
        )

    dtype = choose_dtype()
    result["draft_logits_access"] = test_logits_access(
        model_id=draft_revision.model_id,
        revision=draft_revision.revision,
        tokenizer=draft_tokenizer,
        dtype=dtype,
    )
    result["target_logits_access"] = test_logits_access(
        model_id=target_revision.model_id,
        revision=target_revision.revision,
        tokenizer=target_tokenizer,
        dtype=dtype,
    )
    result["preflight_status"] = "passes_kaggle_preflight"
    result["trace_collection_allowed"] = True
except PreflightFailure as error:
    result["failure"] = {"code": error.code, "message": safe_error_message(error)}
except Exception as error:
    result["failure"] = {
        "code": "unexpected_preflight_failure",
        "message": safe_error_message(error),
    }
finally:
    result["environment"] = environment_metadata()
    persisted_path = write_result(result)

print(
    canonical_json(
        {
            "preflight_status": result["preflight_status"],
            "trace_collection_allowed": result["trace_collection_allowed"],
            "result_path": str(persisted_path),
            "result_sha256": sha256_file(persisted_path),
            "failure_code": None if result["failure"] is None else result["failure"]["code"],
        }
    )
)

if result["preflight_status"] != "passes_kaggle_preflight":
    raise RuntimeError(
        "Kaggle preflight failed. The result file was retained; do not collect traces."
    )

print("Kaggle preflight passed. Stop here; trace collection is a later governed slice.")

## Required retained output

Download this exact file after a successful or failed run:

```text
/kaggle/working/specsafe_v5_qwen_preflight_result.json
```

A pass authorizes only the next controlled trace-collection design slice. It does not authorize a benchmark claim, public dataset release, or production deployment.
